# Full activation-vector experiments: TC

This companion extends `tc_gpt_small_tokens_places.ipynb`.
The original notebook is preserved. These experiments keep every
coordinate and amplitude of the source token codes: no top-feature
selection, minimum-positive replacement, or atomic reduction.

Six source occurrences, two query operations, and four relative
activation levels are fixed in the shared implementation. Layers
0, 8, and 11 run sequentially. The Rio/de pair adds one meet family.
All 25,600 cached rows contribute to every extent and statistic.
Models, tokens, and activation matrices are loaded offline.

In [1]:
import os
import sys
from pathlib import Path

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / 'pyproject.toml').is_file()
)
paper = root / 'texs/sparsesurrs/lattcontexts'
sys.path.insert(0, str(paper / 'python'))
from lattcontextstools.activation_experiments_codexgen import (
    ALPHAS, CASES, OUT, run,
)
import pandas as pd

## Construction and interpretation

A row represents one 128-token corpus block. Its vector is the
coordinatewise maximum of all token activations in that block.
For selected source positions, the full-vector meet keeps the
smallest shared amplitude in each coordinate; the full-vector join
keeps the largest amplitude at any selected position.

At each level alpha in {0.25, 0.5, 0.75, 1}, a row qualifies if
its summary is at least alpha times the query in every coordinate.
Alpha=1 preserves the actual source amplitudes. The separate
support-only baseline requires positive activation and is exactly
a classical binary-context query. A zero meet retrieves all rows.

Six source rows are refreshed in the working matrix to remove small
numerical inconsistencies between freshly computed token codes and
cached sequence maxima. The other 25,594 rows are unchanged.
Original caches are never overwritten. The reports record every
refresh, input SHA-256 hashes, software versions, and code hash.

Counts include the source; subtract one for other retrieved rows.
The lexical reference uses explicit case-insensitive phrase or
whole-word rules. Lexical coincidence is not semantic correctness.
Statistics describe this fixed corpus and these selected examples;
there is no random-sample or held-out generalization claim.

In [2]:
pd.DataFrame([
    dict(case=name, source_row=row, positions=positions,
         lexical_patterns=patterns)
    for name, (row, positions, patterns) in CASES.items()
])

,case,source_row,positions,lexical_patterns
0,nyc,3457,"[1, 2, 3]",[\bnew\s+york\s+city\b]
1,animals,4042,"[8, 82]","[\bcats?\b, \bdogs?\b]"
2,rio,5411,"[15, 16, 17]",[\brio\s+de\s+janeiro\b]
3,richmond,117,"[30, 31]",[\brichmond\s+hill\b]
4,park,21481,"[24, 25, 26]",[\bnational\s+park\s+service\b]
5,sports,1924,"[15, 39, 94]","[\bcleveland\b, \bclippers\b, \bcavaliers\b]"


## Layer 0: execution and statistics

Execute the full protocol for this layer. The returned rows
cover full-vector meets, joins, and the Rio/de pair. Alpha
`None` (displayed as NaN) denotes the support-only
baseline, never alpha zero.
Closure is computed from all sequence summaries at levels
0.5 and 1; its extent is checked against the original extent.
Each extent, query, and checked closed intent is saved in NPZ.

In [3]:
report_0 = run('tc', 0)
frame_0 = pd.json_normalize(report_0['records'])
frame_0[[
    'case', 'operation', 'alpha', 'active_coordinates',
    'count', 'non_source_count', 'retained_support',
    'lexical.both', 'lexical_hit_fraction',
    'lexical_coverage', 'closure_added_coordinates',
]]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2-small into HookedTransformer
Moving model to device:  cpu


nyc 3457 [1, 2, 3] [' New', ' York', ' City']


animals 4042 [8, 82] [' Cat', ' dog']


rio 5411 [15, 16, 17] [' Rio', ' de', ' Janeiro']


...

tc 0 park join 47 [1, 1, 1, 1, 1]


tc 0 sports meet 0 [25600, 25600, 25600, 25600, 25600]


tc 0 sports join 168 [1, 1, 1, 1, 1]


Saved tc_layer0 65 queries


,case,operation,alpha,active_coordinates,count,non_source_count,retained_support,lexical.both,lexical_hit_fraction,lexical_coverage,closure_added_coordinates
0,nyc,meet,NaN,4,7209,7208,1.000000,20,0.002774,0.363636,NaN
1,nyc,meet,0.25,4,5781,5780,0.801914,20,0.003460,0.363636,NaN
2,nyc,meet,0.50,4,4607,4606,0.639062,17,0.003690,0.309091,22.0
3,nyc,meet,0.75,4,3568,3567,0.494937,17,0.004765,0.309091,NaN
4,nyc,meet,1.00,4,2750,2749,0.381468,15,0.005455,0.272727,25.0
...,...,...,...,...,...,...,...,...,...,...,...
60,sports,join,NaN,168,1,0,1.000000,1,1.000000,1.000000,NaN
61,sports,join,0.25,168,1,0,1.000000,1,1.000000,1.000000,NaN
62,sports,join,0.50,168,1,0,1.000000,1,1.000000,1.000000,777.0
63,sports,join,0.75,168,1,0,1.000000,1,1.000000,1.000000,NaN


## Layer 8: execution and statistics

Execute the full protocol for this layer. The returned rows
cover full-vector meets, joins, and the Rio/de pair. Alpha
`None` (displayed as NaN) denotes the support-only
baseline, never alpha zero.
Closure is computed from all sequence summaries at levels
0.5 and 1; its extent is checked against the original extent.
Each extent, query, and checked closed intent is saved in NPZ.

In [4]:
report_8 = run('tc', 8)
frame_8 = pd.json_normalize(report_8['records'])
frame_8[[
    'case', 'operation', 'alpha', 'active_coordinates',
    'count', 'non_source_count', 'retained_support',
    'lexical.both', 'lexical_hit_fraction',
    'lexical_coverage', 'closure_added_coordinates',
]]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2-small into HookedTransformer
Moving model to device:  cpu
nyc 3457 [1, 2, 3] [' New', ' York', ' City']


animals 4042 [8, 82] [' Cat', ' dog']


rio 5411 [15, 16, 17] [' Rio', ' de', ' Janeiro']


richmond 117 [30, 31] [' Richmond', ' Hill']


...

tc 8 park join 171 [1, 1, 1, 1, 1]


tc 8 sports meet 2 [5713, 3899, 2740, 1912, 1349]


tc 8 sports join 160 [1, 1, 1, 1, 1]


Saved tc_layer8 65 queries


,case,operation,alpha,active_coordinates,count,non_source_count,retained_support,lexical.both,lexical_hit_fraction,lexical_coverage,closure_added_coordinates
0,nyc,meet,NaN,1,20837,20836,1.000000,48,0.002304,0.872727,NaN
1,nyc,meet,0.25,1,14673,14672,0.704180,35,0.002385,0.636364,NaN
2,nyc,meet,0.50,1,9197,9196,0.441378,19,0.002066,0.345455,3.0
3,nyc,meet,0.75,1,5776,5775,0.277199,11,0.001904,0.200000,NaN
4,nyc,meet,1.00,1,4184,4183,0.200797,6,0.001434,0.109091,3.0
...,...,...,...,...,...,...,...,...,...,...,...
60,sports,join,NaN,160,1,0,1.000000,1,1.000000,1.000000,NaN
61,sports,join,0.25,160,1,0,1.000000,1,1.000000,1.000000,NaN
62,sports,join,0.50,160,1,0,1.000000,1,1.000000,1.000000,2341.0
63,sports,join,0.75,160,1,0,1.000000,1,1.000000,1.000000,NaN


## Layer 11: execution and statistics

Execute the full protocol for this layer. The returned rows
cover full-vector meets, joins, and the Rio/de pair. Alpha
`None` (displayed as NaN) denotes the support-only
baseline, never alpha zero.
Closure is computed from all sequence summaries at levels
0.5 and 1; its extent is checked against the original extent.
Each extent, query, and checked closed intent is saved in NPZ.

In [5]:
report_11 = run('tc', 11)
frame_11 = pd.json_normalize(report_11['records'])
frame_11[[
    'case', 'operation', 'alpha', 'active_coordinates',
    'count', 'non_source_count', 'retained_support',
    'lexical.both', 'lexical_hit_fraction',
    'lexical_coverage', 'closure_added_coordinates',
]]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2-small into HookedTransformer
Moving model to device:  cpu


nyc 3457 [1, 2, 3] [' New', ' York', ' City']


animals 4042 [8, 82] [' Cat', ' dog']


rio 5411 [15, 16, 17] [' Rio', ' de', ' Janeiro']


...

tc 11 park join 308 [1, 1, 1, 1, 1]


tc 11 sports meet 1 [801, 667, 559, 480, 420]


tc 11 sports join 329 [1, 1, 1, 1, 1]


Saved tc_layer11 65 queries


,case,operation,alpha,active_coordinates,count,non_source_count,retained_support,lexical.both,lexical_hit_fraction,lexical_coverage,closure_added_coordinates
0,nyc,meet,NaN,5,6794,6793,1.000000,19,0.002797,0.345455,NaN
1,nyc,meet,0.25,5,4484,4483,0.659994,13,0.002899,0.236364,NaN
2,nyc,meet,0.50,5,2910,2909,0.428319,9,0.003093,0.163636,54.0
3,nyc,meet,0.75,5,1887,1886,0.277745,7,0.003710,0.127273,NaN
4,nyc,meet,1.00,5,1203,1202,0.177068,5,0.004156,0.090909,65.0
...,...,...,...,...,...,...,...,...,...,...,...
60,sports,join,NaN,329,1,0,1.000000,1,1.000000,1.000000,NaN
61,sports,join,0.25,329,1,0,1.000000,1,1.000000,1.000000,NaN
62,sports,join,0.50,329,1,0,1.000000,1,1.000000,1.000000,2264.0
63,sports,join,0.75,329,1,0,1.000000,1,1.000000,1.000000,NaN


## Reading the outcomes

A shrinking extent across positive alpha levels is evidence that
magnitudes affect retrieval while the active-coordinate pattern
stays fixed. A constant extent does not demonstrate that effect.
A one-row extent retrieves only the source. A zero meet is a
degenerate query, not evidence for a shared semantic feature.

The manuscript compares these sets with the other surrogate using
all four cells of a coincidence table and Jaccard overlap. Only
corpus row identities are compared across models; latent feature
indices and activation units are not assumed to be aligned.

In [6]:
all_records = pd.concat([frame_0, frame_8, frame_11], ignore_index=True)
print('Evaluated extents:', len(all_records))
print('All closure and lattice-identity assertions passed.')
print('Results directory:', OUT)

Evaluated extents: 195
All closure and lattice-identity assertions passed.
Results directory: /Users/ltsinadze/git/my_papers/texs/sparsesurrs/lattcontexts/experiments/activation_results


## Query type and token-witness presentation

These are source-token meets
and joins. Their cached contextual subset at blocks 8 and 11 is displayed
in `contextual_witnesses_codexgen.ipynb`, using all-coordinate inspection
and a yellow D badge for distributed satisfaction. The external category
galleries instead use meets of pooled document vectors.